<i>Copyright (c) Recommenders contributors.</i>

<i>Licensed under the MIT License.</i>

# NPA: Neural News Recommendation with Personalized Attention
NPA \[1\] is a news recommendation model with personalized attention. The core of NPA is a news representation model and a user representation model. In the news representation model we use a CNN network to learn hidden representations of news articles based on their titles. In the user representation model we learn the representations of users based on the representations of their clicked news articles. In addition, a word-level and a news-level personalized attention are used to capture different informativeness for different users.

## Properties of NPA:
- NPA is a content-based news recommendation method.
- It uses a CNN network to learn news representation. And it learns user representations from their clicked news articles.
- A word-level personalized attention is used to help NPA attend to important words for different users.
- A news-level personalized attention is used to help NPA attend to important historical clicked news for different users.

## Data format:
For quicker training and evaluaiton, we sample MINDdemo dataset of 5k users from [MIND small dataset](https://msnews.github.io/). The MINDdemo dataset has the same file format as MINDsmall and MINDlarge. If you want to try experiments on MINDsmall
 and MINDlarge, please change the dowload source.
 Select the MIND_type parameter from ['large', 'small', 'demo'] to choose dataset.
 
**MINDdemo_train** is used for training, and **MINDdemo_dev** is used for evaluation. Training data and evaluation data are composed of a news file and a behaviors file. You can find more detailed data description in [MIND repo](https://github.com/msnews/msnews.github.io/blob/master/assets/doc/introduction.md)

### news data
This file contains news information including newsid, category, subcatgory, news title, news abstarct, news url and entities in news title, entities in news abstarct.
One simple example: <br>

`N46466	lifestyle	lifestyleroyals	The Brands Queen Elizabeth, Prince Charles, and Prince Philip Swear By	Shop the notebooks, jackets, and more that the royals can't live without.	https://www.msn.com/en-us/lifestyle/lifestyleroyals/the-brands-queen-elizabeth,-prince-charles,-and-prince-philip-swear-by/ss-AAGH0ET?ocid=chopendata	[{"Label": "Prince Philip, Duke of Edinburgh", "Type": "P", "WikidataId": "Q80976", "Confidence": 1.0, "OccurrenceOffsets": [48], "SurfaceForms": ["Prince Philip"]}, {"Label": "Charles, Prince of Wales", "Type": "P", "WikidataId": "Q43274", "Confidence": 1.0, "OccurrenceOffsets": [28], "SurfaceForms": ["Prince Charles"]}, {"Label": "Elizabeth II", "Type": "P", "WikidataId": "Q9682", "Confidence": 0.97, "OccurrenceOffsets": [11], "SurfaceForms": ["Queen Elizabeth"]}]	[]`
<br>

In general, each line in data file represents information of one piece of news: <br>

`[News ID] [Category] [Subcategory] [News Title] [News Abstrct] [News Url] [Entities in News Title] [Entities in News Abstract] ...`

<br>

We generate a word_dict file to tranform words in news title to word indexes, and a embedding matrix is initted from pretrained glove embeddings.

### behaviors data
One simple example: <br>
`1	U82271	11/11/2019 3:28:58 PM	N3130 N11621 N12917 N4574 N12140 N9748	N13390-0 N7180-0 N20785-0 N6937-0 N15776-0 N25810-0 N20820-0 N6885-0 N27294-0 N18835-0 N16945-0 N7410-0 N23967-0 N22679-0 N20532-0 N26651-0 N22078-0 N4098-0 N16473-0 N13841-0 N15660-0 N25787-0 N2315-0 N1615-0 N9087-0 N23880-0 N3600-0 N24479-0 N22882-0 N26308-0 N13594-0 N2220-0 N28356-0 N17083-0 N21415-0 N18671-0 N9440-0 N17759-0 N10861-0 N21830-0 N8064-0 N5675-0 N15037-0 N26154-0 N15368-1 N481-0 N3256-0 N20663-0 N23940-0 N7654-0 N10729-0 N7090-0 N23596-0 N15901-0 N16348-0 N13645-0 N8124-0 N20094-0 N27774-0 N23011-0 N14832-0 N15971-0 N27729-0 N2167-0 N11186-0 N18390-0 N21328-0 N10992-0 N20122-0 N1958-0 N2004-0 N26156-0 N17632-0 N26146-0 N17322-0 N18403-0 N17397-0 N18215-0 N14475-0 N9781-0 N17958-0 N3370-0 N1127-0 N15525-0 N12657-0 N10537-0 N18224-0`
<br>

In general, each line in data file represents one instance of an impression. The format is like: <br>

`[Impression ID] [User ID] [Impression Time] [User Click History] [Impression News]`

<br>

User Click History is the user historical clicked news before Impression Time. Impression News is the displayed news in an impression, which format is:<br>

`[News ID 1]-[label1] ... [News ID n]-[labeln]`

<br>
Label represents whether the news is clicked by the user. All information of news in User Click History and Impression News can be found in news data file.

## Global settings and imports

In [1]:
MODEL_NAME = 'npa'
import os, sys, pickle, re, zipfile, tensorflow as tf, numpy as np, pandas as pd
from tqdm import tqdm
import os, sys
sys.path.insert(0, os.getcwd())
from recommenders.models.deeprec.deeprec_utils import download_deeprec_resources, prepare_hparams
from recommenders.models.newsrec.newsrec_utils import get_mind_data_set
from recommenders.models.newsrec.models.npa import NPAModel
from recommenders.models.newsrec.io.mind_iterator import MINDIterator
from recommenders.utils.notebook_utils import store_metadata

# Path Configuration
base_path = os.getcwd()
data_path = os.path.join(base_path, 'data')
utils_path = os.path.join(base_path, 'utils')

train_news_file = os.path.join(data_path, 'train', 'news.tsv')
train_behaviors_file = os.path.join(data_path, 'train', 'behaviors.tsv')
valid_news_file = os.path.join(data_path, 'valid', 'news.tsv')
valid_behaviors_file = os.path.join(data_path, 'valid', 'behaviors.tsv')
wordEmb_file = os.path.join(utils_path, 'embedding_all.npy')
userDict_file = os.path.join(utils_path, 'uid2index.pkl')
wordDict_file = os.path.join(utils_path, 'word_dict_all.pkl')
vertDict_file = os.path.join(utils_path, 'vert_dict.pkl')
subvertDict_file = os.path.join(utils_path, 'subvert_dict.pkl')
yaml_file = os.path.join(utils_path, f'{MODEL_NAME}.yaml')

print(f'Setup complete for {MODEL_NAME.upper()}')


Setup complete for NPA


## Prepare Parameters

In [2]:
epochs = 1
seed = 42
batch_size = 64

## Download and load data

In [3]:
PREPARATION_MODE = 'official'
MIRROR_URL = 'https://huggingface.co/datasets/Recommenders/MIND/resolve/main/'
if PREPARATION_MODE == 'official' and not os.path.exists(wordEmb_file):
    download_deeprec_resources(MIRROR_URL, utils_path, 'MINDlarge_utils.zip')

if not os.path.exists(wordDict_file):
    print('Building dictionaries...')
    def clean(t): return re.sub(r'[^a-zA-Z0-9\s]', '', t.lower()) if isinstance(t, str) else ''
    words, verts, subverts, users = set(), set(), set(), set()
    for f in [train_news_file, valid_news_file]:
        if os.path.exists(f):
            df = pd.read_csv(f, sep='\t', header=None, quoting=3, names=['id', 'vert', 'subvert', 'title', 'abstract', 'url', 't_ent', 'a_ent'])
            verts.update(df['vert'].dropna()); subverts.update(df['subvert'].dropna())
            for t in tqdm(df['title'].dropna()): words.update(clean(t).split())
    wd = {w: i + 1 for i, w in enumerate(sorted(list(words)))}; wd['<pad>'] = 0
    vd = {v: i + 1 for i, v in enumerate(sorted(list(verts)))}
    svd = {s: i + 1 for i, s in enumerate(sorted(list(subverts)))}
    if os.path.exists(train_behaviors_file):
        df_b = pd.read_csv(train_behaviors_file, sep='\t', header=None, names=['imp', 'uid', 'time', 'his', 'imps'])
        users.update(df_b['uid'].dropna())
    u2i = {u: i + 1 for i, u in enumerate(sorted(list(users)))}
    for p, d in [(wordDict_file, wd), (vertDict_file, vd), (subvertDict_file, svd), (userDict_file, u2i)]: 
        with open(p, 'wb') as f: pickle.dump(d, f)
    print('Dictionaries ready.')


## Create hyper-parameters

In [4]:
hparams = prepare_hparams(yaml_file, 
                          wordEmb_file=wordEmb_file,
                          wordDict_file=wordDict_file, 
                          userDict_file=userDict_file,
                          batch_size=batch_size,
                          epochs=epochs)

# Load dictionaries if not in memory (fixes NameError: vd)
import pickle
if 'vd' not in locals():
    with open(vertDict_file, 'rb') as f: vd = pickle.load(f)
if 'svd' not in locals():
    with open(subvertDict_file, 'rb') as f: svd = pickle.load(f)

hparams.vert_num = len(vd) + 1
hparams.subvert_num = len(svd) + 1
print(hparams)

import pickle


import pickle


HParams object with values {'use_entity': True, 'use_context': True, 'cross_activation': 'identity', 'user_dropout': False, 'dropout': [0.2], 'attention_dropout': 0.0, 'load_saved_model': False, 'fast_CIN_d': 0, 'use_Linear_part': False, 'use_FM_part': False, 'use_CIN_part': False, 'use_DNN_part': False, 'init_method': 'tnormal', 'init_value': 0.01, 'embed_l2': 0.0, 'embed_l1': 0.0, 'layer_l2': 0.0, 'layer_l1': 0.0, 'cross_l2': 0.0, 'cross_l1': 0.0, 'reg_kg': 0.0, 'learning_rate': 0.0001, 'lr_rs': 1, 'lr_kg': 0.5, 'kg_training_interval': 5, 'max_grad_norm': 2, 'is_clip_norm': 0, 'dtype': 32, 'optimizer': 'adam', 'epochs': 1, 'batch_size': 64, 'enable_BN': False, 'show_step': 100000, 'save_model': True, 'save_epoch': 5, 'write_tfevents': False, 'train_num_ngs': 4, 'need_sample': True, 'embedding_dropout': 0.0, 'EARLY_STOP': 100, 'min_seq_length': 1, 'slots': 5, 'cell': 'SUM', 'title_size': 10, 'his_size': 50, 'data_format': 'news', 'npratio': 4, 'attention_hidden_dim': 200, 'word_emb_di

In [5]:
iterator = MINDIterator

## Train the NPA model

In [6]:
model = NPAModel(hparams, iterator, seed=seed)

2026-04-07 14:31:13.884640: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4 Max
2026-04-07 14:31:13.884664: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 36.00 GB
2026-04-07 14:31:13.884670: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 14.04 GB
2026-04-07 14:31:13.884694: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-04-07 14:31:13.884707: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
2026-04-07 14:31:13.916722: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:388] MLIR V1 optimization pass is not enabled
2026-04-07 14:31:13.917420: I tensorflow/core/gr

In [7]:
print(model.run_eval(valid_news_file, valid_behaviors_file))

0it [00:00, ?it/s]/opt/anaconda3/envs/CRT/lib/python3.11/site-packages/keras/src/engine/training_v1.py:2359: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates=self.state_updates,
2026-04-07 14:31:15.807176: W tensorflow/c/c_api.cc:305] Operation '{name:'dense_3/bias/Assign' id:582 op device:{requested: '', assigned: ''} def:{{{node dense_3/bias/Assign}} = AssignVariableOp[_has_manual_control_dependencies=true, dtype=DT_FLOAT, validate_shape=false](dense_3/bias, dense_3/bias/Initializer/zeros)}}' was changed by setting attribute after it was run by a session. This mutation will have no effect, and will trigger an error in the future. Either don't modify nodes after running them or create a new session.
42829it [22:38, 31.53it/s]


{'group_auc': 0.5034, 'mean_mrr': 0.2218, 'ndcg@5': 0.2259, 'ndcg@10': 0.2916}


In [8]:
%%time
model.fit(train_news_file, train_behaviors_file, valid_news_file, valid_behaviors_file)

0it [00:00, ?it/s]2026-04-07 14:54:22.807244: W tensorflow/c/c_api.cc:305] Operation '{name:'loss/mul' id:1290 op device:{requested: '', assigned: ''} def:{{{node loss/mul}} = Mul[T=DT_FLOAT, _has_manual_control_dependencies=true](loss/mul/x, loss/activation_2_loss/value)}}' was changed by setting attribute after it was run by a session. This mutation will have no effect, and will trigger an error in the future. Either don't modify nodes after running them or create a new session.
2026-04-07 14:54:22.867437: W tensorflow/c/c_api.cc:305] Operation '{name:'training/Adam/dense_3/bias/v/Assign' id:1888 op device:{requested: '', assigned: ''} def:{{{node training/Adam/dense_3/bias/v/Assign}} = AssignVariableOp[_has_manual_control_dependencies=true, dtype=DT_FLOAT, validate_shape=false](training/Adam/dense_3/bias/v, training/Adam/dense_3/bias/v/Initializer/zeros)}}' was changed by setting attribute after it was run by a session. This mutation will have no effect, and will trigger an error in

at epoch 1
train info: logloss loss:0.46152996460521567
eval info: group_auc:0.5044, mean_mrr:0.2177, ndcg@10:0.2862, ndcg@5:0.2242
at epoch 1 , train time: 656.6 eval time: 1438.2
CPU times: user 16min 57s, sys: 28min 25s, total: 45min 22s
Wall time: 34min 54s


In [9]:
%%time
res_syn = model.run_eval(valid_news_file, valid_behaviors_file)
print(res_syn)

42829it [23:18, 30.63it/s]


{'group_auc': 0.5044, 'mean_mrr': 0.2177, 'ndcg@5': 0.2242, 'ndcg@10': 0.2862}
CPU times: user 6min 39s, sys: 11min 18s, total: 17min 57s
Wall time: 23min 45s


In [10]:
# Record results for tests - ignore this cell
store_metadata("group_auc", res_syn['group_auc'])
store_metadata("mean_mrr", res_syn['mean_mrr'])
store_metadata("ndcg@5", res_syn['ndcg@5'])
store_metadata("ndcg@10", res_syn['ndcg@10'])

## Save the model

In [11]:
model_path = os.path.join(data_path, "model")
os.makedirs(model_path, exist_ok=True)

model.model.save_weights(os.path.join(model_path, "npa_ckpt"))

## Output Predcition File
This code segment is used to generate the prediction.zip file, which is in the same format in [MIND Competition Submission Tutorial](https://competitions.codalab.org/competitions/24122#learn_the_details-submission-guidelines).

Please change the `MIND_type` parameter to `large` if you want to submit your prediction to [MIND Competition](https://msnews.github.io/competition.html).

In [12]:
group_impr_indexes, group_labels, group_preds = model.run_slow_eval(valid_news_file, valid_behaviors_file)

42829it [22:57, 31.10it/s]


In [13]:
with open(os.path.join(data_path, 'prediction.txt'), 'w') as f:
    for impr_index, preds in tqdm(zip(group_impr_indexes, group_preds)):
        impr_index += 1
        pred_rank = (np.argsort(np.argsort(preds)[::-1]) + 1).tolist()
        pred_rank = '[' + ','.join([str(i) for i in pred_rank]) + ']'
        f.write(' '.join([str(impr_index), pred_rank])+ '\n')

73152it [00:00, 151404.45it/s]


In [14]:
f = zipfile.ZipFile(os.path.join(data_path, 'prediction.zip'), 'w', zipfile.ZIP_DEFLATED)
f.write(os.path.join(data_path, 'prediction.txt'), arcname='prediction.txt')
f.close()

## Reference
\[1\] Chuhan Wu, Fangzhao Wu, Mingxiao An, Jianqiang Huang, Yongfeng Huang and Xing Xie: NPA: Neural News Recommendation with Personalized Attention, KDD 2019, ADS track.<br>
\[2\] Wu, Fangzhao, et al. "MIND: A Large-scale Dataset for News Recommendation" Proceedings of the 58th Annual Meeting of the Association for Computational Linguistics. https://msnews.github.io/competition.html <br>
\[3\] GloVe: Global Vectors for Word Representation. https://nlp.stanford.edu/projects/glove/